# Bashorat — modelni Colab/Kaggle GPU'da o'qitish

Bu notebook butun quvurni ishga tushiradi:
1. 12 yillik narxlarni yuklash (forex, kripto, aksiyalar, bug'doy, neft, oltin...)
2. Ochiq yangilik datasetlari (Kaggle DJIA, FNSPID, GDELT)
3. FinBERT (muzlatilgan) bilan embedding → kunlik jamlash (leakage himoyasi bilan)
4. Walk-forward baholash + ablation (narx / +mavsum / +bozorlar / +yangilik), Transformer v2 + ansambl
5. Final holdout — **faqat oxirida, bir marta**
6. Servis modelini saqlash va serverga ko'chirish

`Runtime → Change runtime type → T4 GPU` ni tanlang.

In [ ]:
# Loyihani yuklang: atlas.zip ni Colab'ga yuklang (chap paneldagi Files) yoki GitHub'dan clone qiling
import os
if not os.path.exists("atlas"):
    !unzip -q atlas.zip -d .
%cd atlas
!pip install -q -r requirements.txt

In [ ]:
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Narxlar (Yahoo Finance, 12 yil, kunlik)

In [ ]:
!python -m atlas.cli download --years 12

## 2. Yangilik datasetlari
**Kaggle DJIA** (2008–2016, faqat sana — ertasi kundan ishlatiladi):
Kaggle → Account → Create API token → `kaggle.json` ni yuklang.

In [ ]:
os.makedirs("data/news/raw", exist_ok=True)
if os.path.exists("/content/kaggle.json") or os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    !mkdir -p ~/.kaggle && cp -n /content/kaggle.json ~/.kaggle/ 2>/dev/null; chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d aaron7sun/stocknews -p data/news/raw --unzip
else:
    print("kaggle.json topilmadi — Kaggle dataseti o'tkazib yuborildi")

**FNSPID** (moliyaviy yangiliklar, vaqt belgisi bilan; fayl katta ~ bir necha GB):

In [ ]:
USE_FNSPID = True
if USE_FNSPID:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download("Zihan1004/FNSPID", "Stock_news/nasdaq_exteral_data.csv", repo_type="dataset",
                        local_dir="data/news/raw")
    !mv data/news/raw/Stock_news/nasdaq_exteral_data.csv data/news/raw/ 2>/dev/null; ls -lh data/news/raw

**GDELT** — so'nggi 90 kun (DOC API). Tarixiy GDELT uchun `atlas/data/news_open.py` dagi `GDELT_BQ_SQL` ni BigQuery'da ishga tushirib, natijani `data/news/raw/gdelt_bq.csv` ga qo'ying.

In [ ]:
!python -m atlas.cli gdelt --days 90

## 3. FinBERT embedding + yangilik kubi

In [ ]:
!python -m atlas.cli features --embedder finbert

## 4. Walk-forward baholash va ablation
Holdoutga tegilmaydi. Natijaga qarab feature tanlamang — aks holda testga moslashtirib qo'yasiz.

In [ ]:
!python -m atlas.cli evaluate --device cuda

In [ ]:
import json
r = json.load(open("data/reports/report_walkforward.json"))
import pandas as pd
display(pd.DataFrame(r["walk_forward_summary"]).T)
display(pd.DataFrame(r["ablation_tests"]).T)

## 5. Final holdout (faqat bir marta, hamma qarorlar qabul qilingandan keyin)

In [ ]:
# !python -m atlas.cli evaluate --final --device cuda

### Backtest (faqat holdout, out-of-sample)

In [ ]:
# !python -m atlas.cli backtest --threshold 0.05 --cost-bps 5

## 6. Servis modeli va yuklab olish

In [ ]:
!python -m atlas.cli train --device cuda
!cd data && zip -qr ../atlas_data.zip prices embeddings models reports
from google.colab import files; files.download("atlas_data.zip")

Serverda: `atlas_data.zip` ni loyiha ichidagi `data/` papkasiga oching va `python -m atlas.cli serve` ni ishga tushiring.